In [ ]:
import os

In [ ]:
print(os.getcwd())

In [ ]:
from youtube import extract_recommendation_json
from youtube import extract_video_ids
from youtube import lookup_metadata_yt_api

In [4]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:

#Keep Playwright outside of the async loop here
!python youtube/extract_recommendation_json.py

In [ ]:
!python youtube/extract_video_ids.py

In [ ]:
!python youtube/lookup_metadata_yt_api.py

Looking up 10 videos...
Saved videos.json


In [ ]:
import json

with open("outputs/videos.json", "r", encoding="utf-8") as f:
    videos = json.load(f)

for video in videos:
    print(f"Title: {video['title']}")
    print(f"Channel: {video['channel']}")
    print(f"Views: {video['views']}")
    print(f"Duration: {video['duration']}")
    print(f"Video ID: {video['videoId']}")
    print()

In [ ]:
# import json
# import pandas as pd

# with open("outputs/videos.json", "r", encoding="utf-8") as f:
#     videos = json.load(f)

# df = pd.DataFrame(videos)

# df[:50]

In [ ]:
# df["description"]

# df["description"].to_csv('output_file.csv', index=False)

In [ ]:
# from youtube_transcript_api import YouTubeTranscriptApi
# ytt_api = YouTubeTranscriptApi()
# ytt_api.fetch("1e6n6RZeTWQ")

In [ ]:
# from dotenv import load_dotenv
# import os

# load_dotenv(override=True)


In [6]:
# Integrated from embedding_llm.py

from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
from embedding_report import detailed_report
import tiktoken
import numpy as np

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

# --------------------------------------------------
# Split text into overlapping chunks
# --------------------------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)

    chunks = []

    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunks.append(tokenizer.decode(chunk_tokens))

    return chunks


# --------------------------------------------------
# Create embeddings
# --------------------------------------------------
def embed_text(text):
    chunks = chunk_text(text)

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunks
    )

    embeddings = [item.embedding for item in response.data]

    return chunks, embeddings


# --------------------------------------------------
# Cosine similarity
# --------------------------------------------------
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

#Avg similarity score
def average_similarity(essay_embeddings, transcript_embeddings):
    total_similarity = 0
    comparisons = 0

    for essay_vec in essay_embeddings:
        for transcript_vec in transcript_embeddings:
            total_similarity += cosine_similarity(essay_vec, transcript_vec)
            comparisons += 1

    average = total_similarity / comparisons

    print(f"Average Similarity: {average:.4f}")

    return average


# --------------------------------------------------
# Compare every chunk
# --------------------------------------------------
def compare_embeddings(
    essay_chunks,
    essay_embeddings,
    transcript_chunks,
    transcript_embeddings,
    top_k=10
):
    results = []

    for essay_idx, essay_vec in enumerate(essay_embeddings):
        for transcript_idx, transcript_vec in enumerate(transcript_embeddings):

            similarity = cosine_similarity(
                essay_vec,
                transcript_vec
            )

            results.append({
                "essay_chunk": essay_idx,
                "transcript_chunk": transcript_idx,
                "similarity": similarity
            })

    results.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    print(f"\nTop {top_k} Matches")
    print("=" * 80)

    for result in results[:top_k]:

        e = result["essay_chunk"]
        t = result["transcript_chunk"]

        print(f"\nSimilarity: {result['similarity']:.4f}")
        print("-" * 80)

        print(f"Essay Chunk #{e}")
        print(essay_chunks[e])

        print("\nTranscript Chunk #{}".format(t))
        print(transcript_chunks[t])

        print("=" * 80)

    return results


# --------------------------------------------------
# Download transcript
# --------------------------------------------------
ytt_api = YouTubeTranscriptApi()


#Scores an average of 0.45 similarity score
video_id = "1e6n6RZeTWQ"

#Rammatra lore (completely unrelated example)]
#Similarity score of 0.2 - 0.35
#GUess the common theme is speaking out against cultures that has affected the speaker
# video_id = "kQ_y0QcorJ8"

# fetched_transcript = ytt_api.fetch(video_id)

# transcript = " ".join(
#     snippet.text for snippet in fetched_transcript
# )

transcript = """Key Pillars of Letting Go
1. The Art of Non-Doing (Wu Wei)

Wu Wei translates to effortless action, non-doing, or entering a flow state.

While some control is necessary for survival, over-controlling (like micro-managing relationships, future outcomes, or children) interrupts natural processes. Just as you cannot force a tree to grow faster by pulling on it, things like attraction, trust, and healing must be allowed to unfold spontaneously.

2. Embracing Change

Life is a constant dance of opposites (yin and yang). People who cling rigidly to their circumstances or constantly swim against the stream waste massive amounts of energy and live in misery.

Lao Tzu noted that the living are soft and yielding while the dead are rigid and stiff. Embracing change means being flexible, accepting how things are, and adapting to what is useful in the present moment rather than fighting reality.

3. Not Focusing on Outcomes

Fixating on future results (such as status, wealth, or winning a prize) breeds anxiety and paralyzes us in the present.

When an archer focuses too much on an external reward rather than the act of shooting itself, their performance suffers. True flow happens when you are so immersed in the present task that future outcomes fade away.

4. Letting Go of Excess

Society conditions us to chase the top, but the tallest trees catch the most wind, and maintaining high status requires exhausting effort and competition.

By recognizing what we genuinely need—just as a bird needs only one branch to nest—we can let go of unnecessary possessions that act as prison cells, allowing us to travel light and live sustainably."""


# --------------------------------------------------
# Your essay
# --------------------------------------------------
essay = """
1. The Redefinition of Strength
Society's Warped Ideal: Society conditions men to strive toward physical strength as the ultimate measure of worth. However, trauma distorts this definition into a defense mechanism rather than genuine fortitude.

Takezo vs. Otsu: Takezo views strength as emotional invulnerability—armoring himself so he feels no pain, grief, fear, or guilt. In contrast, Otsu possesses true strength because she remains vulnerable and emotionally open, even when her feelings cause her to lose her temper.

True Strength: Real strength is the capacity to change something within or outside yourself to resolve an unfavorable outcome. Because Takezo uses "invincibility" to run away from his emotions rather than accept them, his ideal actively undermines his ability to bear pain.

2. The Feedback Loop of the Ego
Social Proof and Isolation: Takezo's brutal lethality and the fear it instills in villagers create a feedback loop. His kills provide personal and social proof that his "invincible" armor works, continually feeding and reinforcing his ego.

The Shattering Moment: Takezo cannot realize his delusion while actively fighting. It is only when he is captured and strung up to a tree—forced into absolute stillness—that his ego is stripped of its outlet. Unmoving, he is forced to confront his buried trauma: the pain of being abandoned by his mother and shunned by his father.

3. The Crisis of Identity and Despair
Unprocessed Primary Emotions: Facing his memories floods Takezo with unprocessed primary emotions—grief over failing his invincible ideal, fear of dying without warrior validation, and latent guilt for the lives he ended.

The Emptiness of Proving Himself: When facing a potential death at the hands of Takuan, the temporary ego boost of standing his ground brings no lasting relief. He is left entirely empty. His striving was meaningless, leaving behind only a man in deep, inescapable pain.

Despair and Self-Destruction: Overwhelmed by the sudden flood of suppressed suffering, he attempts to bash his head against a rock, finding self-destruction easier than feeling his raw vulnerability.

4. The Ego's Trap and Final Awakening
Secondary Emotion as Armor: When threatened, the ego uses secondary emotions (like anger and aggression) to avoid primary pain. It constructs grand ideals to ensure this avoidance remains permanent.

The Cost of Failure: Once you are prevented from pursuing your idealized shield, the suppressed emotions inevitably return. Takezo believed invincibility would save him from the pain of living, but it only misled him—leaving him to question his entire existence once the illusion collapsed.
"""


# --------------------------------------------------
# Create embeddings
# --------------------------------------------------
essay_chunks, essay_embeddings = embed_text(essay)

transcript_chunks, transcript_embeddings = embed_text(transcript)


# --------------------------------------------------
# Compare them
# --------------------------------------------------
results = compare_embeddings(
    essay_chunks,
    essay_embeddings,
    transcript_chunks,
    transcript_embeddings,
    top_k=10
)

average_score = average_similarity(
    essay_embeddings,
    transcript_embeddings
)


Top 10 Matches

Similarity: 0.6006
--------------------------------------------------------------------------------
Essay Chunk #0

Societal Shift in Control: Modern society has transitioned from a punitive system demanding obedience to an achievement-based system encouraging limitless potential, shifting the source of depression from external restriction to the pressure of tying self-worth to accomplishments.

The Dynamics of "Auto-Exploitation": Unlike punishment, which requires costly external enforcement, achievement-driven control relies on individuals policing themselves—using self-loathing and despair as internal motivators to work continuously while maintaining the illusion of personal freedom.

Personal Experience with Performative Self-Improvement: High school efforts—such as optimizing study habits for extra game time, learning to code due to economic trends, and studying happiness/social skills to gain approval—were driven by external incentives rather than genuine intrins

In [ ]:
video_summary = """ Here is a bullet point summary of the video "Why Being 'Behind' in your 20s is Brilliant." by Aleks:

Reframing "Being Behind": While societal norms treat falling off the traditional success path as a failure, stepping outside external expectations provides a valuable opportunity to define personal priorities and build a genuinely fulfilling life [00:15].

Breaking the Need for Academic Validation: Early academic achievement can act as a distraction from genuine self-worth; losing that external validation forces one to build real self-appreciation rooted in character growth and empathy rather than performance [13:33].

Life Disruptions as Opportunities to Pivot: Unforeseen challenges—such as health issues or personal trauma—can disrupt ideal trajectories, offering a chance to reevaluate desires rather than forcing oneself onto an ill-fitting path [08:15].

Reevaluating Societal Success Metrics: Conventional definitions of success (e.g., high-prestige careers or high income) often require high-stress environments that may harm well-being, highlighting the value of prioritizing health, work-life balance, and personal alignment instead [12:07].

Avoiding the Midlife Crisis: Falling behind in one's 20s allows for early self-discovery, helping prevent the identity crises that occur later in life when people blindly follow traditional paths without questioning what they truly want [17:21]."""

In [ ]:
# # --------------------------------------------------
# # Compare the essay against the first 10 recommended video transcripts
# # --------------------------------------------------

# import json
# from embedding_report import detailed_report

# with open("outputs/video_ids.json", "r", encoding="utf-8") as f:
#     recommended_video_ids = json.load(f)[:5]

# with open("outputs/videos.json", "r", encoding="utf-8") as f:
#     video_metadata = json.load(f)

# video_titles = {
#     video["videoId"]: video["title"]
#     for video in video_metadata
# }

# essay_chunks, essay_embeddings = embed_text(essay)
# scores = []

# for video_number, video_id in enumerate(recommended_video_ids, start=1):
#     video_title = video_titles.get(video_id, "Unknown title")

#     try:
#         fetched_transcript = ytt_api.fetch(video_id)
#         transcript = " ".join(
#             snippet.text for snippet in fetched_transcript
#         )

#         if not transcript.strip():
#             print(f"Skipping {video_title} ({video_id}): transcript is empty")
#             continue

#         transcript_chunks, transcript_embeddings = embed_text(transcript)
#         average_score = detailed_report(
#             essay_chunks,
#             essay_embeddings,
#             transcript_chunks,
#             transcript_embeddings,
#             video_id,
#             video_title,
#         )

#         scores.append({
#             "video_id": video_id,
#             "title": video_title,
#             "average_similarity": average_score
#         })
#         print(f"Compared {video_number}/10: {video_title} ({video_id})")

#     except Exception as error:
#         print(f"Skipping {video_title} ({video_id}): {error}")

# scores.sort(
#     key=lambda result: result["average_similarity"],
#     reverse=True
# )

# print("\nRanked average similarity scores:")
# for result in scores:
#     print(
#         f"{result['title']} ({result['video_id']}): "
#         f"{result['average_similarity']:.4f}"
#     )


In [ ]:
# 